In [1]:
import sys, os, json, gc
from pathlib import Path
from collections import Counter

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print("python:", sys.executable)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

PROJECT = Path(r"C:\Users\shlok\projects\ddp-llm\parser")
DATA = PROJECT / "data"
CHECKPOINTS = PROJECT / "checkpoints"

ADAPTER_1P5B = CHECKPOINTS / "qwen1.5b-qlora-v1" / "checkpoint-240"
ADAPTER_3B   = CHECKPOINTS / "qwen3b-qlora-v1" / "checkpoint-240"

assert ADAPTER_1P5B.exists(), f"missing: {ADAPTER_1P5B}"
assert ADAPTER_3B.exists(), f"missing: {ADAPTER_3B}"
assert (DATA / "test.jsonl").exists()
assert (DATA / "train_v1.jsonl").exists()
print("all paths ok")

python: C:\Users\shlok\projects\ddp-llm\.venv\Scripts\python.exe
torch: 2.11.0+cu128
cuda available: True
gpu: NVIDIA GeForce RTX 5060 Laptop GPU
all paths ok


In [2]:
SYSTEM_PROMPT = """You are a parser that converts a user's natural language response into a subset of the shown options.

The user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.

Output format (JSON only, nothing else):
- A JSON list of the favored labels, e.g. ["A", "B"]
- [] if the user explicitly rejects ALL options ("none of these", "all wrong")
- "*" if the utterance is off-topic OR expresses no usable preference ("I don't know", "they all look the same", "I love football")

Rules:
- Any positive signal about an option means it goes in the list.
- "X is better than Y" endorses only X, not Y.
- "X and Y are both good, X is better" endorses both X and Y.
- Negations like "not D" or "anything but B" mean the remaining options go in the list.
- Questions like "is it A?" are treated as tentative endorsement of A.

Output ONLY the JSON. No explanation, no prose."""


def load_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding="utf-8").splitlines() if l.strip()]


def build_model(base_model_id, adapter_path):
    """Load base in 4-bit, attach LoRA adapter, return (model, tokenizer)."""
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    tok = AutoTokenizer.from_pretrained(base_model_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    base = AutoModelForCausalLM.from_pretrained(
        base_model_id, quantization_config=bnb, device_map="cuda",
    )
    m = PeftModel.from_pretrained(base, str(adapter_path))
    m.eval()
    return m, tok


def parse(model, tok, utterance, options):
    """Run parser on one utterance. Returns (parsed, raw_text)."""
    options_str = "[" + ", ".join(f'"{o}"' for o in options) + "]"
    user_msg = f"Options: {options_str}\nUser: {utterance}"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    inputs = tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=15, do_sample=False,
            pad_token_id=tok.eos_token_id,
        )
    raw = tok.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True,
    ).strip()
    if raw.startswith("```"):
        raw = raw.strip("`").lstrip("json").strip()
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        return None, raw
    return parsed, raw


def labels_equal(a, b):
    if a == "*" or b == "*":
        return a == b
    if isinstance(a, list) and isinstance(b, list):
        return set(a) == set(b)
    return False


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    print(f"VRAM after cleanup (GB): {torch.cuda.memory_allocated() / 1e9:.2f}")


print("helpers defined")

helpers defined


In [3]:
# 2-option adversarial cases — this is what the app calls parse() with.
# The key failures: ambiguous/no-signal utterances that should map to "*"
# but the parser confidently outputs ["A"] or ["B"] instead.
ADVERSARIAL_TWO_OPT = [
    # -- ambiguous / no-signal (should all be "*") --
    ("works great",    "*"),
    ("looks nice",     "*"),
    ("hmm",            "*"),
    ("okay",           "*"),
    ("meh",            "*"),
    ("not sure",       "*"),
    ("dunno really",   "*"),
    ("cool",           "*"),
    ("sure whatever",  "*"),
    ("interesting",    "*"),
    ("i guess",        "*"),
    ("perfect",        "*"),
    ("nice one",       "*"),
    ("go with it",     "*"),
    ("that works",     "*"),
    # -- sanity checks (should still work) --
    ("A",              ["A"]),
    ("B",              ["B"]),
    ("the left one",   ["A"]),
    ("the right one",  ["B"]),
    ("neither",        []),
    ("none of them",   []),
    ("both are fine",  ["A", "B"]),
    ("A and B both work", ["A", "B"]),
]

n_ambig = sum(1 for _, g in ADVERSARIAL_TWO_OPT if g == "*")
n_sanity = len(ADVERSARIAL_TWO_OPT) - n_ambig
print(f"adversarial set: {len(ADVERSARIAL_TWO_OPT)} cases ({n_ambig} ambiguous, {n_sanity} sanity)")

adversarial set: 23 cases (15 ambiguous, 8 sanity)


In [4]:
test_data = load_jsonl(DATA / "test.jsonl")
print(f"loaded {len(test_data)} test examples")
print("categories:", Counter(ex.get("category", "unknown") for ex in test_data))


def eval_test_set(model, tok, test_data, label):
    """Standard 4-option held-out test set. Sanity check we still hit ~99%."""
    correct = 0
    per_cat = {}
    failures = []
    for ex in test_data:
        parsed, raw = parse(model, tok, ex["utterance"], ex["options"])
        ok = labels_equal(parsed, ex["label"])
        cat = ex.get("category", "unknown")
        per_cat.setdefault(cat, [0, 0])
        per_cat[cat][1] += 1
        if ok:
            correct += 1
            per_cat[cat][0] += 1
        else:
            failures.append((ex, parsed, raw))
    total = len(test_data)
    print(f"\n=== {label}: standard test set ===")
    print(f"overall: {correct}/{total} = {100*correct/total:.1f}%")
    for cat in sorted(per_cat):
        c, t = per_cat[cat]
        print(f"  {cat:20s} {c}/{t} = {100*c/t:.1f}%")
    if failures:
        print(f"failures ({len(failures)}):")
        for ex, parsed, raw in failures[:10]:
            print(f"  utt={ex['utterance']!r}")
            print(f"    gold={ex['label']!r}  got={parsed!r}  raw={raw!r}")
    return correct, total, failures


def eval_adversarial(model, tok, label):
    """2-option adversarial — the important test."""
    print(f"\n=== {label}: 2-option adversarial ===")
    n_correct_ambig = 0
    n_correct_sanity = 0
    n_ambig = 0
    n_sanity = 0
    for utt, gold in ADVERSARIAL_TWO_OPT:
        parsed, raw = parse(model, tok, utt, ("A", "B"))
        ok = labels_equal(parsed, gold)
        is_ambig = (gold == "*")
        if is_ambig:
            n_ambig += 1
            if ok:
                n_correct_ambig += 1
        else:
            n_sanity += 1
            if ok:
                n_correct_sanity += 1
        mark = "OK  " if ok else "FAIL"
        tag = "AMB" if is_ambig else "SAN"
        print(f"  [{mark}] [{tag}] {utt!r:22s} gold={gold!r:14s} got={parsed!r}")
    print(f"\nambiguous:  {n_correct_ambig}/{n_ambig} = {100*n_correct_ambig/n_ambig:.1f}%")
    print(f"sanity:     {n_correct_sanity}/{n_sanity} = {100*n_correct_sanity/n_sanity:.1f}%")
    return n_correct_ambig, n_ambig, n_correct_sanity, n_sanity

loaded 210 test examples
categories: Counter({'negation': 61, 'reject_all': 29, 'mixed_sentiment': 28, 'comparative': 26, 'uncertainty': 22, 'off_topic': 22, 'multi_positive': 13, 'single_positive': 9})


In [5]:
print("="*60)
print("loading 1.5B...")
print("="*60)
model, tok = build_model("Qwen/Qwen2.5-1.5B-Instruct", ADAPTER_1P5B)
print(f"VRAM after load (GB): {torch.cuda.memory_allocated() / 1e9:.2f}")

results_1p5b_test = eval_test_set(model, tok, test_data, "1.5B")
results_1p5b_adv  = eval_adversarial(model, tok, "1.5B")

del model, tok
free_gpu()

loading 1.5B...


Task was destroyed but it is pending!
task: <Task pending name='Task-276' coro=<_async_in_context.<locals>.run_in_context() done, defined at C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-277' coro=<Kernel.shell_main() running at C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>
C:\Users\shlok\AppData\Local\Python\pythoncore-3.12-64\Lib\re\_parser.py:261: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  def get(self):
Task was destroyed but it is pending!
task: <Task pending name='Task-277' coro=<Kernel.shell_main() running at C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]>
W0902 15:40:45.550000 47248 Lib\site-packages\torch\utils\flop_counter.p

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


VRAM after load (GB): 1.23


C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



=== 1.5B: standard test set ===
overall: 207/210 = 98.6%
  comparative          25/26 = 96.2%
  mixed_sentiment      27/28 = 96.4%
  multi_positive       12/13 = 92.3%
  negation             61/61 = 100.0%
  off_topic            22/22 = 100.0%
  reject_all           29/29 = 100.0%
  single_positive      9/9 = 100.0%
  uncertainty          22/22 = 100.0%
failures (3):
  utt='B leaves D in the dust'
    gold=['B']  got=['A', 'C']  raw='["A", "C"]'
  utt='A perfect, B terrible, C okay, D terrible'
    gold=['A', 'C']  got=['A']  raw='["A"]'
  utt='A is okay but B is closer'
    gold=['A', 'B']  got=['B']  raw='["B"]'

=== 1.5B: 2-option adversarial ===
  [FAIL] [AMB] 'works great'          gold='*'            got=['A']
  [OK  ] [AMB] 'looks nice'           gold='*'            got='*'
  [OK  ] [AMB] 'hmm'                  gold='*'            got='*'
  [OK  ] [AMB] 'okay'                 gold='*'            got='*'
  [OK  ] [AMB] 'meh'                  gold='*'            got='*'
  [OK  ] 

In [6]:
print("="*60)
print("loading 3B...")
print("="*60)
model, tok = build_model("Qwen/Qwen2.5-3B-Instruct", ADAPTER_3B)
print(f"VRAM after load (GB): {torch.cuda.memory_allocated() / 1e9:.2f}")

results_3b_test = eval_test_set(model, tok, test_data, "3B")
results_3b_adv  = eval_adversarial(model, tok, "3B")

del model, tok
free_gpu()

loading 3B...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

VRAM after load (GB): 2.19

=== 3B: standard test set ===
overall: 207/210 = 98.6%
  comparative          26/26 = 100.0%
  mixed_sentiment      27/28 = 96.4%
  multi_positive       12/13 = 92.3%
  negation             60/61 = 98.4%
  off_topic            22/22 = 100.0%
  reject_all           29/29 = 100.0%
  single_positive      9/9 = 100.0%
  uncertainty          22/22 = 100.0%
failures (3):
  utt='the fourth is the only one'
    gold=['D']  got=['A', 'B', 'C']  raw='["A", "B", "C"]'
  utt='A perfect, B terrible, C okay, D terrible'
    gold=['A', 'C']  got=['A']  raw='["A"]'
  utt='A is okay but B is closer'
    gold=['A', 'B']  got=['B']  raw='["B"]'

=== 3B: 2-option adversarial ===
  [FAIL] [AMB] 'works great'          gold='*'            got=['A']
  [FAIL] [AMB] 'looks nice'           gold='*'            got=['A']
  [OK  ] [AMB] 'hmm'                  gold='*'            got='*'
  [OK  ] [AMB] 'okay'                 gold='*'            got='*'
  [OK  ] [AMB] 'meh'                

In [7]:
train_data = load_jsonl(DATA / "train_v1.jsonl")
print(f"train: {len(train_data)}, test: {len(test_data)}")
print()

print("=== train label distribution ===")
def label_kind(l):
    if l == "*": return "star"
    if l == []: return "empty"
    if isinstance(l, list) and len(l) == 1: return "single"
    if isinstance(l, list) and len(l) == 2: return "two"
    if isinstance(l, list) and len(l) >= 3: return "three_plus"
    return "other"
print(Counter(label_kind(ex["label"]) for ex in train_data))
print()

print("=== train categories ===")
print(Counter(ex.get("category", "unknown") for ex in train_data))
print()

# The critical one: how many "*" examples are there, and what do they look like?
star_examples = [ex for ex in train_data if ex["label"] == "*"]
print(f"=== '*' examples in train: {len(star_examples)} ===")
for ex in star_examples[:15]:
    print(f"  {ex['utterance']!r}  (cat={ex.get('category', '?')})")
print(f"  ... ({len(star_examples)-15} more)" if len(star_examples) > 15 else "")
print()

# And: are there any training examples that look like "works great"?
# i.e., short positive utterances with no explicit letter/ordinal reference?
print("=== short positive utterances in train (heuristic: <20 chars, contains 'good/great/perfect/nice/love/like') ===")
positive_words = ["great", "perfect", "nice", "love", "like", "cool", "awesome", "amazing"]
short_positives = [
    ex for ex in train_data
    if len(ex["utterance"]) < 20
    and any(w in ex["utterance"].lower() for w in positive_words)
]
for ex in short_positives[:20]:
    print(f"  {ex['utterance']!r:40s} -> {ex['label']!r}  (cat={ex.get('category', '?')})")
print(f"total: {len(short_positives)}")

train: 1278, test: 210

=== train label distribution ===
Counter({'two': 339, 'single': 272, 'star': 264, 'three_plus': 228, 'empty': 175})

=== train categories ===
Counter({'negation': 368, 'reject_all': 175, 'mixed_sentiment': 171, 'comparative': 158, 'off_topic': 132, 'uncertainty': 132, 'multi_positive': 82, 'single_positive': 60})

=== '*' examples in train: 264 ===
  'who won the game last night'  (cat=off_topic)
  'no strong opinion'  (cat=uncertainty)
  'cant separate them'  (cat=uncertainty)
  'petrol has gone up again'  (cat=off_topic)
  'internet is down'  (cat=off_topic)
  'not clear which is which'  (cat=uncertainty)
  'my neighbor is playing loud music'  (cat=off_topic)
  'blur, cant tell'  (cat=uncertainty)
  'need to charge my phone'  (cat=off_topic)
  'no lean either way'  (cat=uncertainty)
  'no idea?'  (cat=uncertainty)
  'need coffee'  (cat=off_topic)
  'not sure at all'  (cat=uncertainty)
  'i just cant tell them apart'  (cat=uncertainty)
  'delhi traffic is crazy

In [8]:
# 180 new examples across 4 new/extended categories to fix the failure modes
# discovered in the adversarial diagnostic.

NEW_EXAMPLES = []

# ==========================================================================
# ambient_positive (60): short positive reactions with NO referent -> "*"
# This is the missing pattern that caused "works great" -> ["A"] failures.
# ==========================================================================
AMBIENT_POSITIVE = [
    "works great", "works well", "that works", "that works for me",
    "that'll work", "this works", "it works", "works fine",
    "perfect", "perfect!", "absolutely perfect", "just perfect",
    "great", "great!", "sounds great", "looks great", "looks nice",
    "nice", "nice one", "nice!", "very nice", "quite nice",
    "cool", "cool!", "cool cool", "pretty cool", "sounds cool",
    "excellent", "brilliant", "solid", "sweet", "sweet!",
    "amazing", "awesome", "wonderful", "fantastic", "lovely",
    "yeah nice", "yeah cool", "yeah works", "oh nice", "oh cool",
    "go with it", "go for it", "go ahead", "sounds good", "sounds fine",
    "fine by me", "all good", "looks good", "looks fine", "seems fine",
    "seems good", "seems okay", "okay looks alright", "not bad",
    "not bad at all", "quite good", "pretty good", "decent",
    "sure looks fine", "yeah works fine",
]
for utt in AMBIENT_POSITIVE:
    NEW_EXAMPLES.append({
        "utterance": utt,
        "options": ["A", "B", "C", "D"],
        "label": "*",
        "category": "ambient_positive",
    })

# ==========================================================================
# ambient_negative (30): bare negative reactions with NO referent -> "*"
# Distinct from reject_all ("none of these"); these just express affect.
# ==========================================================================
AMBIENT_NEGATIVE = [
    "eh", "meh", "bleh", "hmm", "hm", "ehh",
    "boring", "bland", "dull", "lame", "kinda dull",
    "not great", "not really", "not feeling it", "meh not feeling it",
    "not impressed", "underwhelming", "disappointing",
    "yawn", "whatever", "uninteresting", "so-so", "meh whatever",
    "weak", "kinda weak", "kinda meh", "not exciting",
    "nothing special", "nothing great", "not sold",
]
for utt in AMBIENT_NEGATIVE:
    NEW_EXAMPLES.append({
        "utterance": utt,
        "options": ["A", "B", "C", "D"],
        "label": "*",
        "category": "ambient_negative",
    })

# ==========================================================================
# position_words (50): left/right/first/last/top/bottom that DO refer to
# a specific option. Mix of N=2, 3, 4 to teach the model position words
# resolve against option COUNT, not against training distribution.
# Especially: "the right one" -> right-hand side, not "the correct one".
# ==========================================================================
POSITION_2OPT = [
    ("the left one", ["A"]), ("left one", ["A"]), ("go left", ["A"]),
    ("the one on the left", ["A"]), ("left please", ["A"]),
    ("yeah left", ["A"]), ("i pick the left", ["A"]),
    ("the right one", ["B"]), ("right one", ["B"]), ("go right", ["B"]),
    ("the one on the right", ["B"]), ("right please", ["B"]),
    ("yeah right", ["B"]), ("i pick the right", ["B"]),
    ("the first one", ["A"]), ("first please", ["A"]),
    ("the second one", ["B"]), ("second please", ["B"]),
]
for utt, lab in POSITION_2OPT:
    NEW_EXAMPLES.append({
        "utterance": utt,
        "options": ["A", "B"],
        "label": lab,
        "category": "position_words",
    })

POSITION_3OPT = [
    ("the left one", ["A"]), ("the middle one", ["B"]), ("the right one", ["C"]),
    ("go with the middle", ["B"]), ("middle please", ["B"]),
    ("first one", ["A"]), ("the last one", ["C"]),
    ("second one", ["B"]), ("third please", ["C"]),
    ("the one on the far right", ["C"]),
]
for utt, lab in POSITION_3OPT:
    NEW_EXAMPLES.append({
        "utterance": utt,
        "options": ["A", "B", "C"],
        "label": lab,
        "category": "position_words",
    })

POSITION_4OPT = [
    ("the leftmost", ["A"]), ("the rightmost", ["D"]),
    ("the one on the far left", ["A"]), ("the one on the far right", ["D"]),
    ("second from the left", ["B"]), ("second from the right", ["C"]),
    ("the middle two", ["B", "C"]), ("the two on the right", ["C", "D"]),
    ("the two on the left", ["A", "B"]),
    ("first and last", ["A", "D"]), ("first and second", ["A", "B"]),
    ("third and fourth", ["C", "D"]),
    ("top one", ["A"]), ("bottom one", ["D"]),
]
for utt, lab in POSITION_4OPT:
    NEW_EXAMPLES.append({
        "utterance": utt,
        "options": ["A", "B", "C", "D"],
        "label": lab,
        "category": "position_words",
    })

# ==========================================================================
# multi_positive_extra (40): more "both are fine" / "either works" / "any"
# variants. The 3B regressed on "both are fine" -> "*" so we reinforce.
# ==========================================================================
MULTI_2OPT = [
    ("both are fine", ["A", "B"]), ("both work", ["A", "B"]),
    ("both good", ["A", "B"]), ("both are good", ["A", "B"]),
    ("either works", ["A", "B"]), ("either one", ["A", "B"]),
    ("either is fine", ["A", "B"]), ("any of them", ["A", "B"]),
    ("both look right", ["A", "B"]), ("either would do", ["A", "B"]),
    ("both A and B work", ["A", "B"]), ("A and B both fine", ["A", "B"]),
    ("yes to both", ["A", "B"]), ("both please", ["A", "B"]),
    ("they both look good", ["A", "B"]),
]
for utt, lab in MULTI_2OPT:
    NEW_EXAMPLES.append({
        "utterance": utt,
        "options": ["A", "B"],
        "label": lab,
        "category": "multi_positive",
    })

MULTI_3OPT = [
    ("all three work", ["A", "B", "C"]), ("all of them", ["A", "B", "C"]),
    ("any of these three", ["A", "B", "C"]), ("A and B please", ["A", "B"]),
    ("A and C", ["A", "C"]), ("B and C both fine", ["B", "C"]),
    ("first two are fine", ["A", "B"]), ("last two work", ["B", "C"]),
]
for utt, lab in MULTI_3OPT:
    NEW_EXAMPLES.append({
        "utterance": utt,
        "options": ["A", "B", "C"],
        "label": lab,
        "category": "multi_positive",
    })

MULTI_4OPT = [
    ("all four work", ["A", "B", "C", "D"]),
    ("any of these four", ["A", "B", "C", "D"]),
    ("all of them are fine", ["A", "B", "C", "D"]),
    ("A B and C", ["A", "B", "C"]),
    ("A B and D", ["A", "B", "D"]),
    ("B C and D", ["B", "C", "D"]),
    ("A and D both work", ["A", "D"]),
    ("B and D please", ["B", "D"]),
    ("first three are fine", ["A", "B", "C"]),
    ("last three are fine", ["B", "C", "D"]),
    ("A C and D look right", ["A", "C", "D"]),
    ("all of A B C", ["A", "B", "C"]),
    ("yes to A and D", ["A", "D"]),
    ("yes to B C D", ["B", "C", "D"]),
    ("all except A", ["B", "C", "D"]),
    ("all except D", ["A", "B", "C"]),
    ("both B and C", ["B", "C"]),
]
for utt, lab in MULTI_4OPT:
    NEW_EXAMPLES.append({
        "utterance": utt,
        "options": ["A", "B", "C", "D"],
        "label": lab,
        "category": "multi_positive",
    })

print(f"total new examples: {len(NEW_EXAMPLES)}")
print(f"by category: {Counter(ex['category'] for ex in NEW_EXAMPLES)}")
print(f"by label kind: {Counter(label_kind(ex['label']) for ex in NEW_EXAMPLES)}")

Task was destroyed but it is pending!
task: <Task pending name='Task-549' coro=<_async_in_context.<locals>.run_in_context() done, defined at C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-550' coro=<Kernel.shell_main() running at C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>
C:\Users\shlok\AppData\Local\Python\pythoncore-3.12-64\Lib\collections\__init__.py:449: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  result = tuple_new(cls, iterable)
Task was destroyed but it is pending!
task: <Task pending name='Task-550' coro=<Kernel.shell_main() running at C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]>
Task was destroyed but it is pending!
task:

total new examples: 174
by category: Counter({'ambient_positive': 62, 'position_words': 42, 'multi_positive': 40, 'ambient_negative': 30})
by label kind: Counter({'star': 92, 'single': 36, 'two': 30, 'three_plus': 16})


In [9]:
# Check for duplicate utterances against existing train.jsonl
existing_utts = {ex["utterance"].lower().strip() for ex in train_data}
new_utts = [ex["utterance"] for ex in NEW_EXAMPLES]
dupes = [u for u in new_utts if u.lower().strip() in existing_utts]

print(f"duplicates with existing train.jsonl: {len(dupes)}")
for d in dupes:
    print(f"  {d!r}")

# Also check duplicates within the new set itself (same utterance different options is OK)
from collections import defaultdict
by_utt = defaultdict(list)
for ex in NEW_EXAMPLES:
    by_utt[ex["utterance"]].append(ex)
internal_dupes = {u: v for u, v in by_utt.items() if len(v) > 1}
print(f"\ninternal duplicates (same utterance, may have different options): {len(internal_dupes)}")
for u, exs in internal_dupes.items():
    print(f"  {u!r}")
    for ex in exs:
        print(f"    opts={ex['options']}  label={ex['label']}")

# Random sample to eyeball
import random
random.seed(0)
print("\n=== random sample of new examples ===")
for ex in random.sample(NEW_EXAMPLES, 12):
    print(f"  cat={ex['category']:20s} opts={ex['options']}  utt={ex['utterance']!r}  label={ex['label']}")

duplicates with existing train.jsonl: 1
  'A and C'

internal duplicates (same utterance, may have different options): 3
  'the left one'
    opts=['A', 'B']  label=['A']
    opts=['A', 'B', 'C']  label=['A']
  'the right one'
    opts=['A', 'B']  label=['B']
    opts=['A', 'B', 'C']  label=['C']
  'the one on the far right'
    opts=['A', 'B', 'C']  label=['C']
    opts=['A', 'B', 'C', 'D']  label=['D']

=== random sample of new examples ===
  cat=position_words       opts=['A', 'B']  utt='i pick the left'  label=['A']
  cat=position_words       opts=['A', 'B']  utt='first please'  label=['A']
  cat=ambient_positive     opts=['A', 'B', 'C', 'D']  utt='absolutely perfect'  label=*
  cat=ambient_negative     opts=['A', 'B', 'C', 'D']  utt='hm'  label=*
  cat=position_words       opts=['A', 'B', 'C', 'D']  utt='first and second'  label=['A', 'B']
  cat=position_words       opts=['A', 'B', 'C', 'D']  utt='second from the left'  label=['B']
  cat=position_words       opts=['A', 'B']  utt='

In [10]:
import shutil
from datetime import datetime

# drop the one duplicate
NEW_EXAMPLES_CLEAN = [
    ex for ex in NEW_EXAMPLES
    if ex["utterance"].lower().strip() not in existing_utts
]
print(f"after dedupe: {len(NEW_EXAMPLES_CLEAN)} new examples (dropped {len(NEW_EXAMPLES) - len(NEW_EXAMPLES_CLEAN)})")

# train_v2.jsonl = train_v1.jsonl + new examples
train_path = DATA / "train_v2.jsonl"
shutil.copy(DATA / "train_v1.jsonl", train_path)

# append the new examples
with train_path.open("a", encoding="utf-8") as f:
    for ex in NEW_EXAMPLES_CLEAN:
        f.write(json.dumps(ex) + "\n")

# verify
train_data_v2 = load_jsonl(train_path)
print(f"\ntrain.jsonl now has: {len(train_data_v2)} examples (was {len(train_data)})")
print(f"delta: +{len(train_data_v2) - len(train_data)}")
print(f"\nnew category distribution:")
print(Counter(ex.get("category", "unknown") for ex in train_data_v2))

Task was destroyed but it is pending!
task: <Task pending name='Task-730' coro=<_async_in_context.<locals>.run_in_context() done, defined at C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-731' coro=<Kernel.shell_main() running at C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>
C:\Users\shlok\AppData\Local\Python\pythoncore-3.12-64\Lib\json\decoder.py:354: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  obj, end = self.scan_once(s, idx)
Task was destroyed but it is pending!
task: <Task pending name='Task-731' coro=<Kernel.shell_main() running at C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]>


after dedupe: 173 new examples (dropped 1)
backed up: train.jsonl.bak-20260902-162518

train.jsonl now has: 1451 examples (was 1278)
delta: +173

new category distribution:
Counter({'negation': 368, 'reject_all': 175, 'mixed_sentiment': 171, 'comparative': 158, 'off_topic': 132, 'uncertainty': 132, 'multi_positive': 121, 'ambient_positive': 62, 'single_positive': 60, 'position_words': 42, 'ambient_negative': 30})


In [5]:
from transformers import BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# reload train data fresh in case cell 10 was re-run
train_data_full = load_jsonl(DATA / "train_v2.jsonl")
print(f"training on {len(train_data_full)} examples")

# format as chat messages (matches inference-time prompt exactly)
def to_chat(ex):
    options_str = "[" + ", ".join(f'"{o}"' for o in ex["options"]) + "]"
    user_msg = f"Options: {options_str}\nUser: {ex['utterance']}"
    assistant_msg = json.dumps(ex["label"])
    return {"messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg},
    ]}

train_ds = Dataset.from_list([to_chat(ex) for ex in train_data_full])
print(f"dataset size: {len(train_ds)}")
print("sample:")
print(train_ds[0])

training on 1451 examples
dataset size: 1451
sample:
{'messages': [{'role': 'system', 'content': 'You are a parser that converts a user\'s natural language response into a subset of the shown options.\n\nThe user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.\n\nOutput format (JSON only, nothing else):\n- A JSON list of the favored labels, e.g. ["A", "B"]\n- [] if the user explicitly rejects ALL options ("none of these", "all wrong")\n- "*" if the utterance is off-topic OR expresses no usable preference ("I don\'t know", "they all look the same", "I love football")\n\nRules:\n- Any positive signal about an option means it goes in the list.\n- "X is better than Y" endorses only X, not Y.\n- "X and Y are both good, X is better" endorses both X and Y.\n- Negations like "not D" or "anything but B" mean the remaining options go in the list.\n- Questions like "is it A?" are treate

In [12]:
free_gpu()

MODEL_1P5B = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_1P5B_V2 = CHECKPOINTS / "qwen1p5b-qlora-v2_smaller"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(MODEL_1P5B)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(
    MODEL_1P5B, quantization_config=bnb, device_map="cuda",
)
base = prepare_model_for_kbit_training(base)

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(base, lora_cfg)
model.print_trainable_parameters()

sft_cfg = SFTConfig(
    output_dir=str(OUTPUT_1P5B_V2),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=3,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    max_length=1024,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    processing_class=tok,
    args=sft_cfg,
)

print("starting training...")
trainer.train()
print("done. saving final adapter...")
trainer.save_model(str(OUTPUT_1P5B_V2 / "checkpoint-final"))
print(f"saved to {OUTPUT_1P5B_V2}")

del model, base, trainer, tok
free_gpu()

VRAM after cleanup (GB): 0.01


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


Tokenizing train dataset:   0%|          | 0/1451 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1451 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


starting training...


Step,Training Loss
10,2.623115
20,0.693454
30,0.083857
40,0.073443
50,0.071710
60,0.069187
70,0.065930
80,0.064197
90,0.063856
100,0.061550


done. saving final adapter...
saved to C:\Users\shlok\projects\ddp-llm\parser\checkpoints\qwen1.5b-qlora-v2
VRAM after cleanup (GB): 0.95


In [5]:
free_gpu()

MODEL_3B = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_3B_V2 = CHECKPOINTS / "qwen3b-qlora-v2"

# rebuild train_ds since kernel restarted
def to_chat(ex):
    options_str = "[" + ", ".join(f'"{o}"' for o in ex["options"]) + "]"
    user_msg = f"Options: {options_str}\nUser: {ex['utterance']}"
    assistant_msg = json.dumps(ex["label"])
    return {"messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg},
    ]}

from datasets import Dataset
train_ds = Dataset.from_list([to_chat(ex) for ex in train_data_full])
print(f"training on {len(train_ds)} examples")

from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(MODEL_3B)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(
    MODEL_3B, quantization_config=bnb, device_map="cuda",
)
base = prepare_model_for_kbit_training(base)

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(base, lora_cfg)
model.print_trainable_parameters()

sft_cfg = SFTConfig(
    output_dir=str(OUTPUT_3B_V2),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=5,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    bf16=False,
    optim="paged_adamw_8bit",
    report_to="none",
    max_length=768,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    processing_class=tok,
    args=sft_cfg,
)

print("starting training...")
trainer.train()
print("done. saving final adapter...")
trainer.save_model(str(OUTPUT_3B_V2 / "checkpoint-final"))
print(f"saved to {OUTPUT_3B_V2}")

del model, base, trainer, tok
free_gpu()

VRAM after cleanup (GB): 0.00
training on 1451 examples


W0902 18:06:05.809000 22852 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


Tokenizing train dataset:   0%|          | 0/1451 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1451 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


starting training...


Step,Training Loss
5,3.799260
10,2.266586
15,1.228878
20,0.232256
25,0.088087
30,0.073579
35,0.072373
40,0.069410
45,0.067653
50,0.068848


done. saving final adapter...
saved to C:\Users\shlok\projects\ddp-llm\parser\checkpoints\qwen3b-qlora-v2
VRAM after cleanup (GB): 1.26


In [ ]:
dir C:\Users\shlok\projects\ddp-llm\parser\checkpoints\qwen3b-qlora-v2

In [ ]:
free_gpu()

ADAPTER_1P5B_V2 = CHECKPOINTS / "qwen1p5b-qlora-v2_smaller" / "checkpoint-final"
assert ADAPTER_1P5B_V2.exists(), f"missing: {ADAPTER_1P5B_V2}"

print("="*60)
print("loading 1.5B v2...")
print("="*60)
model, tok = build_model("Qwen/Qwen2.5-1.5B-Instruct", ADAPTER_1P5B_V2)
print(f"VRAM after load (GB): {torch.cuda.memory_allocated() / 1e9:.2f}")

r_1p5b_v2_test = eval_test_set(model, tok, test_data, "1.5B-v2")
r_1p5b_v2_adv  = eval_adversarial(model, tok, "1.5B-v2")

del model, tok
free_gpu()

VRAM after cleanup (GB): 0.00
loading 1.5B v2...


W0902 19:19:26.185000 29156 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


VRAM after load (GB): 1.23


C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
